# 📊 04. Control Estadístico de Procesos (SPC) Bioestadístico y Reglas de Nelson
**Proyecto**: AgroStats AndTech — Plataforma de Inteligencia Agrícola Colombiana  
**Capa del Lakehouse**: **Gold** (Feature Store `features_spc_stability.parquet`)  
**Técnicas**: Gráficos Shewhart Individual Moving Range ($I-MR$), Cartas $\bar{X}-S$, Reglas de Nelson 1 a 4, Índices de Capacidad $C_p$ y $C_{pk}$  
**Estándares**: ISO 7870 (Control Charts), Bioestadística de Walter A. Shewhart, DAMA-BOK  

---

### Objetivos del Cuaderno:
1. **Límites Shewhart 3-Sigma**: Calcular Línea Central ($\bar{X}$), Desviación ($\sigma$), Límites de Alerta ($2\sigma$) y Límites de Control ($3\sigma$) para precios mayoristas y calidad de fruto.
2. **Motor de Reglas de Nelson (1 a 4)**: Implementar algorítmicamente la detección de eventos especiales:
   - **Regla 1**: Punto más allá de $3\sigma$ (Shock agudo de oferta o helada).
   - **Regla 2**: 9 puntos consecutivos del mismo lado de la media (Desplazamiento estructural del nivel de precios).
   - **Regla 3**: 6 puntos consecutivos en aumento o descenso constante (Tendencia inflacionaria o estacional).
   - **Regla 4**: 14 puntos alternando consecutivamente arriba y abajo (Oscilación o inestabilidad de despacho).
3. **Cartas Bioestadísticas de Calidad**: Evaluar homogeneidad en Grados Brix y Calibre entre lotes cosechados.
4. **Capacidad de Proceso ($C_p, C_{pk}$)**: Determinar si el proceso de cosecha cumple de forma predecible con los requisitos técnicos de los compradores internacionales.


In [1]:
import matplotlib
matplotlib.use('Agg')
# 1. Configuración de Entorno e Importación de Librerías
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

WORKSPACE_DIR = Path.cwd()
if WORKSPACE_DIR.name in ["notebooks", "04_gold_spc_biostatistics"]:
    BASE_DIR = WORKSPACE_DIR.parents[1] if WORKSPACE_DIR.name == "04_gold_spc_biostatistics" else WORKSPACE_DIR.parent
else:
    BASE_DIR = WORKSPACE_DIR

GOLD_SPC_FILE = BASE_DIR / "data" / "gold" / "features" / "features_spc_stability.parquet"
plt.rcParams["figure.figsize"] = (14, 6)
sns.set_theme(style="whitegrid")
print(f"Cargando dataset de Control Estadístico desde: {GOLD_SPC_FILE}")


Cargando dataset de Control Estadístico desde: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\gold\features\features_spc_stability.parquet


## 2. Carga y Estructura de Datos SPC
Cargamos la serie temporal continua procesada en la Capa Gold con las alertas de estabilidad y límites precalculados.


In [2]:
df_spc = pd.read_parquet(GOLD_SPC_FILE)
df_spc["fecha_completa"] = pd.to_datetime(df_spc["fecha_completa"])

# Seleccionar producto y mercado
serie_spc = df_spc[
    (df_spc["codigo_cpc"] == "01211") & 
    (df_spc["mercado_id"] == "CORABASTOS")
].sort_values("fecha_completa").reset_index(drop=True)

print(f"Registros disponibles para control estadístico: {len(serie_spc)}")
serie_spc[[
    "fecha_completa", "precio_promedio", "media_historica", "desviacion_sigma",
    "limite_superior_ucl", "limite_inferior_lcl",
    "alerta_nelson_r1_outlier", "alerta_nelson_r2_cambio_media", "alerta_nelson_r3_tendencia"
]].head()


Registros disponibles para control estadístico: 180


,fecha_completa,precio_promedio,media_historica,desviacion_sigma,limite_superior_ucl,limite_inferior_lcl,alerta_nelson_r1_outlier,alerta_nelson_r2_cambio_media,alerta_nelson_r3_tendencia
0,2026-01-01,4564.37,4953.549056,531.812683,6548.987105,3358.111006,False,False,False
1,2026-01-02,4783.02,4953.549056,531.812683,6548.987105,3358.111006,False,False,False
2,2026-01-03,4552.23,4953.549056,531.812683,6548.987105,3358.111006,False,False,False
3,2026-01-04,4513.30,4953.549056,531.812683,6548.987105,3358.111006,False,False,False
4,2026-01-05,4665.01,4953.549056,531.812683,6548.987105,3358.111006,False,False,False


## 3. Gráfico de Control Individual Shewhart ($I-Chart$) con Bandas Sigma
Visualizamos la serie de precios con las zonas $\pm 1\sigma$, $\pm 2\sigma$ (Advertencia) y $\pm 3\sigma$ (Control).


In [3]:
x_bar = serie_spc["media_historica"].iloc[0]
sigma = serie_spc["desviacion_sigma"].iloc[0]
ucl = serie_spc["limite_superior_ucl"].iloc[0]
lcl = serie_spc["limite_inferior_lcl"].iloc[0]
uwl = x_bar + 2.0 * sigma
lwl = max(0.0, x_bar - 2.0 * sigma)

plt.figure(figsize=(14, 6))
plt.plot(serie_spc["fecha_completa"], serie_spc["precio_promedio"], marker="o", markersize=3.5, color="#1f77b4", label="Precio Promedio Diario", alpha=0.8)
plt.axhline(x_bar, color="green", linestyle="--", linewidth=2, label=f"Línea Central (X̄ = {x_bar:,.0f})")
plt.axhline(ucl, color="red", linestyle="-", linewidth=2, label=f"Límite Superior UCL (+3σ = {ucl:,.0f})")
plt.axhline(lcl, color="red", linestyle="-", linewidth=2, label=f"Límite Inferior LCL (-3σ = {lcl:,.0f})")

# Bandas de advertencia y control
plt.axhline(uwl, color="orange", linestyle=":", label=f"Alerta Superior (+2σ = {uwl:,.0f})")
plt.axhline(lwl, color="orange", linestyle=":", label=f"Alerta Inferior (-2σ = {lwl:,.0f})")

plt.fill_between(serie_spc["fecha_completa"], lwl, uwl, color="orange", alpha=0.08)
plt.fill_between(serie_spc["fecha_completa"], lcl, ucl, color="green", alpha=0.05)

plt.title("Carta Shewhart Individual (I-Chart) de Precios Mayoristas (SIPSA Corabastos - Aguacate Hass)", fontsize=13)
plt.xlabel("Fecha", fontsize=11)
plt.ylabel("Precio ($ COP / kg)", fontsize=11)
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


C:\Users\ADAN\AppData\Local\Temp\ipykernel_34996\3226993971.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Detección Visual de las 4 Reglas de Nelson
Identificamos y resaltamos en color rojo brillante los puntos temporales que violan las 4 Reglas de Nelson.


In [4]:
plt.figure(figsize=(14, 6))
plt.plot(serie_spc["fecha_completa"], serie_spc["precio_promedio"], color="#2c3e50", alpha=0.6, label="Precio Diario")
plt.axhline(x_bar, color="green", linestyle="--", label="Línea Central (X̄)")
plt.axhline(ucl, color="red", linestyle="-", label="Límites 3σ (UCL / LCL)")
plt.axhline(lcl, color="red", linestyle="-")

# Resaltar violaciones por regla
r1_pts = serie_spc[serie_spc["alerta_nelson_r1_outlier"]]
r2_pts = serie_spc[serie_spc["alerta_nelson_r2_cambio_media"]]
r3_pts = serie_spc[serie_spc["alerta_nelson_r3_tendencia"]]
r4_pts = serie_spc[serie_spc["alerta_nelson_r4_oscilacion"]]

if not r1_pts.empty:
    plt.scatter(r1_pts["fecha_completa"], r1_pts["precio_promedio"], color="red", s=80, zorder=5, label="Regla 1: Punto > 3σ (Shock)", marker="X")
if not r2_pts.empty:
    plt.scatter(r2_pts["fecha_completa"], r2_pts["precio_promedio"], color="darkorange", s=50, zorder=5, label="Regla 2: 9 Puntos Mismo Lado (Shift)", marker="s")
if not r3_pts.empty:
    plt.scatter(r3_pts["fecha_completa"], r3_pts["precio_promedio"], color="purple", s=60, zorder=5, label="Regla 3: 6 Puntos en Tendencia", marker="^")
if not r4_pts.empty:
    plt.scatter(r4_pts["fecha_completa"], r4_pts["precio_promedio"], color="brown", s=40, zorder=5, label="Regla 4: 14 Puntos Oscilando", marker="d")

plt.title("Diagnóstico de Estabilidad de Mercado: Detección Automatizada de Reglas de Nelson 1 a 4", fontsize=13)
plt.xlabel("Fecha", fontsize=11)
plt.ylabel("Precio ($ COP / kg)", fontsize=11)
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print(f"Violaciones detectadas en el período analizado:")
print(f"- Regla 1 (Outliers > 3σ): {len(r1_pts)} eventos")
print(f"- Regla 2 (Cambio de nivel / Media desplazada): {len(r2_pts)} observaciones")
print(f"- Regla 3 (Tendencia continua 6d): {len(r3_pts)} observaciones")
print(f"- Regla 4 (Oscilación sistemática 14d): {len(r4_pts)} observaciones")


Violaciones detectadas en el período analizado:
- Regla 1 (Outliers > 3σ): 0 eventos
- Regla 2 (Cambio de nivel / Media desplazada): 145 observaciones
- Regla 3 (Tendencia continua 6d): 0 observaciones
- Regla 4 (Oscilación sistemática 14d): 0 observaciones


C:\Users\ADAN\AppData\Local\Temp\ipykernel_34996\3159908634.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Control Bioestadístico en Cosechas Agrícolas e Índices de Capacidad ($C_p, C_{pk}$)
Evaluamos la variabilidad en **Grados Brix** de lotes de cosecha frente a las especificaciones de exportación:
- **Límite Inferior de Especificación (LSL)**: $10.0^\circ$ Brix (Mínimo exigido en aduana/mercado internacional).
- **Límite Superior de Especificación (USL)**: $18.0^\circ$ Brix.


In [5]:
# Carga de datos de lotes
df_batches = pd.read_parquet(BASE_DIR / "data" / "gold" / "features" / "features_yield_prediction.parquet")
brix_values = df_batches["grados_brix"].values

mu_brix = float(np.mean(brix_values))
sigma_brix = float(np.std(brix_values, ddof=1))

LSL = 10.0
USL = 18.0

# Índices de Capacidad de Proceso
cp = (USL - LSL) / (6.0 * sigma_brix)
cpu = (USL - mu_brix) / (3.0 * sigma_brix)
cpl = (mu_brix - LSL) / (3.0 * sigma_brix)
cpk = min(cpu, cpl)

print("--- Índices de Capacidad del Proceso Agrícola (Grados Brix) ---")
print(f"Media de Proceso (μ): {mu_brix:.2f}° Brix | Desviación Estándar (σ): {sigma_brix:.2f}")
print(f"Límite Inferior (LSL): {LSL:.1f}° Brix | Límite Superior (USL): {USL:.1f}° Brix")
print(f"Índice Cp: {cp:.3f}")
print(f"Índice Cpk: {cpk:.3f}")

if cpk >= 1.33:
    print(">> Diagnóstico: Proceso ALTAMENTE CAPAZ y CENTRADO (Cumple holgadamente el estándar de exportación).")
elif cpk >= 1.00:
    print(">> Diagnóstico: Proceso CAPAZ pero ajustado; requiere monitoreo para prevenir lotes no exportables.")
else:
    print(">> Diagnóstico: Proceso NO CAPAZ (cpk < 1.0); genera producto fuera de especificación exportable.")

# Histograma de Capacidad con Curva Normal
plt.figure(figsize=(10, 5))
count, bins, ignored = plt.hist(brix_values, bins=12, density=True, alpha=0.6, color="teal", edgecolor="black", label="Lotes Reales")
xmin, xmax = plt.xlim()
x_axis = np.linspace(xmin, xmax, 100)
plt.plot(x_axis, stats.norm.pdf(x_axis, mu_brix, sigma_brix), "r-", linewidth=2, label="Distribución Normal Ajustada")

plt.axvline(LSL, color="darkred", linestyle="--", linewidth=2, label=f"LSL ({LSL}°)")
plt.axvline(USL, color="darkred", linestyle="--", linewidth=2, label=f"USL ({USL}°)")
plt.axvline(mu_brix, color="blue", linestyle="-", linewidth=1.5, label=f"Media μ ({mu_brix:.1f}°)")

plt.title("Histograma de Capacidad de Proceso: Grados Brix vs Especificaciones de Exportación", fontsize=12)
plt.xlabel("Grados Brix (°)", fontsize=11)
plt.ylabel("Densidad de Probabilidad", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()


--- Índices de Capacidad del Proceso Agrícola (Grados Brix) ---
Media de Proceso (μ): 13.30° Brix | Desviación Estándar (σ): 2.99
Límite Inferior (LSL): 10.0° Brix | Límite Superior (USL): 18.0° Brix
Índice Cp: 0.446
Índice Cpk: 0.368
>> Diagnóstico: Proceso NO CAPAZ (cpk < 1.0); genera producto fuera de especificación exportable.


C:\Users\ADAN\AppData\Local\Temp\ipykernel_34996\3761460195.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Conclusiones y Aplicación Operativa
1. **Detección Temprana**: El motor de Reglas de Nelson permite emitir alertas preventivas a gerentes de compras y asociaciones de productores ante quiebres de precios o sobreofertas.
2. **Capacidad de Exportación**: El cálculo de $C_{pk}$ en cosecha proporciona un indicador clave de desempeño (KPI) para certificar fincas aptas para comercio exterior.
3. **Integración**: Los parámetros y límites de control quedan disponibles en `features_spc_stability.parquet` para el dashboard analítico y las alertas operacionales.
